<a href="https://colab.research.google.com/github/ElizabethFrankWebb/USRI-2026/blob/main/Individual_Base_Model_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [92]:
import numpy as np
import matplotlib.pyplot as plt
import random
import seaborn as ns

In [93]:
#@title parameters

intial_population = 100
loci = 5
selection = 10
carrying_capacity = 100
optimal_trait_value = 0
mutation_rate = 10**-2
base_reproduction_rate = 1.025
generation = 100 #@param {type:"integer"}
genotypes = np.zeros((intial_population, loci))

In [94]:
# Calculate trait value for all individuals

def calculate_trait_z(individual_genotype):
  trait_value = np.sum(individual_genotype)
  return trait_value
trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])

print("Trait values for all individuals:")
print(trait_values)


Trait values for all individuals:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


In [95]:
def calculate_fitness(trait_values, optimal_trait_value, selection, current_population_size, carrying_capacity, base_reproduction_rate):
  trait_values = np.asarray(trait_values)
  genetic_fitness = np.exp(-(trait_values - optimal_trait_value)**2 / (2 * selection))
  # New density-dependent model for expected offspring count per individual:
  # Ensures an average of 1 offspring at carrying_capacity for genetically optimal individuals,
  # and scales base_reproduction_rate accordingly when population is below or above K.
  expected_offspring_per_individual = genetic_fitness * (base_reproduction_rate - (base_reproduction_rate - 1) * (current_population_size / carrying_capacity))
  # Ensure expected_offspring_per_individual is not negative
  return np.maximum(0, expected_offspring_per_individual)

# For the initial population, the current_population_size is len(trait_values).
population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection, len(trait_values), carrying_capacity, base_reproduction_rate)

print(population_fitness)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1.]


In [96]:
#next generation with asexual mating using poisson distrbution for number of offsring


lambdas = population_fitness

offspring_counts = np.random.poisson(lambdas)

print(offspring_counts)

[0 0 0 1 0 0 1 1 1 1 0 0 1 1 1 1 1 1 1 0 1 2 2 1 2 1 0 1 1 0 0 1 2 0 2 1 0
 0 0 2 0 1 0 0 1 1 0 3 0 2 1 1 2 0 0 0 2 1 2 1 1 0 0 0 1 2 3 1 0 3 2 2 0 0
 1 1 2 1 1 1 1 3 1 0 2 1 2 1 0 1 3 4 1 1 1 0 1 2 1 1]


In [97]:
# introducing mutation

def mutate(individual, mutation_rate):
  mutated_individual = individual.copy()
  mutated_loci_info = [] # Store (locus_index, original_value, new_value)
  for locus in range(len(individual)):
    if np.random.rand() < mutation_rate:
      original_value = individual[locus] # Get original value before mutation
      new_value = np.random.normal(0,1) # Assign new value from normal distribution (mean=0, std=1)
      mutated_individual[locus] = new_value
      mutated_loci_info.append((locus, original_value, new_value))
  return mutated_individual, mutated_loci_info # Return both the mutated individual and info about mutations

In [98]:
# Generate the next generation with mutations
new_gen_genotypes_list = []
offspring_mutation_details = []
offspring_counter = 0

# Ensure intial_population is updated to reflect the current size of genotypes array
current_population_size = len(genotypes)

for parent_idx in range(current_population_size):
    parent_genotype = genotypes[parent_idx]
    num_offspring = offspring_counts[parent_idx]

    for _ in range(num_offspring):
        mutated_individual, mutated_loci_info = mutate(parent_genotype, mutation_rate)
        new_gen_genotypes_list.append(mutated_individual)

        if mutated_loci_info:
            offspring_mutation_details.append({
                'offspring_idx': offspring_counter,
                'parent_idx': parent_idx,
                'mutations': mutated_loci_info
            })
        offspring_counter += 1

# Update the genotypes for the new generation
if new_gen_genotypes_list:
    genotypes = np.array(new_gen_genotypes_list)
    intial_population = len(genotypes) # Update population size for the next generation
else:
    genotypes = np.empty((0, loci)) # Handle case with no offspring
    intial_population = 0

print("--- New Generation Genotypes with Mutations ---")
if intial_population > 0:
    for i, individual_genotype in enumerate(genotypes):
        print(f"Offspring {i}: Genotype = {individual_genotype}")
else:
    print("No offspring were generated in this generation.")

print("\n--- Mutation Details ---")
if offspring_mutation_details:
    for detail in offspring_mutation_details:
        print(f"Offspring {detail['offspring_idx']} (from Parent {detail['parent_idx']}) mutated at:")
        for locus_idx, original_val, new_val in detail['mutations']:
            print(f"  Locus {locus_idx}: Changed from {original_val:.4f} to {new_val:.4f}")
else:
    print("No mutations occurred in this generation.")

# Recalculate trait values for the new generation
if intial_population > 0:
    trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])
    print(f"\nRecalculated Trait values for new generation ({intial_population} individuals):")
    print(trait_values)

    # Recalculate fitness for the new generation
    population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection, intial_population, carrying_capacity, base_reproduction_rate)
    print(f"\nRecalculated Fitness for new generation ({intial_population} individuals):")
    print(population_fitness)
else:
    print("\nCannot recalculate trait values or fitness as no offspring were generated.")

--- New Generation Genotypes with Mutations ---
Offspring 0: Genotype = [0. 0. 0. 0. 0.]
Offspring 1: Genotype = [0. 0. 0. 0. 0.]
Offspring 2: Genotype = [0. 0. 0. 0. 0.]
Offspring 3: Genotype = [0. 0. 0. 0. 0.]
Offspring 4: Genotype = [0. 0. 0. 0. 0.]
Offspring 5: Genotype = [0. 0. 0. 0. 0.]
Offspring 6: Genotype = [0. 0. 0. 0. 0.]
Offspring 7: Genotype = [0. 0. 0. 0. 0.]
Offspring 8: Genotype = [0. 0. 0. 0. 0.]
Offspring 9: Genotype = [0. 0. 0. 0. 0.]
Offspring 10: Genotype = [0. 0. 0. 0. 0.]
Offspring 11: Genotype = [0. 0. 0. 0. 0.]
Offspring 12: Genotype = [0. 0. 0. 0. 0.]
Offspring 13: Genotype = [0. 0. 0. 0. 0.]
Offspring 14: Genotype = [0. 0. 0. 0. 0.]
Offspring 15: Genotype = [0. 0. 0. 0. 0.]
Offspring 16: Genotype = [0. 0. 0. 0. 0.]
Offspring 17: Genotype = [0. 0. 0. 0. 0.]
Offspring 18: Genotype = [0. 0. 0. 0. 0.]
Offspring 19: Genotype = [0. 0. 0. 0. 0.]
Offspring 20: Genotype = [0. 0. 0. 0. 0.]
Offspring 21: Genotype = [0. 0. 0. 0. 0.]
Offspring 22: Genotype = [0. 0. 0. 0. 

### Simulation Loop

In [99]:
fitness_history = [] # Initialize list to store fitness data for each generation

# Calculate and store the fitness of the initial population *before* any generational loops.
# This ensures 'population_fitness' is consistent with 'genotypes' at the very start.
initial_trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])
current_pop_size = len(genotypes) # Initial population size
population_fitness = calculate_fitness(initial_trait_values, optimal_trait_value, selection, current_pop_size, carrying_capacity, base_reproduction_rate)
fitness_history.append(population_fitness.copy()) # Store fitness of the initial population

for g in range(generation):
    # Check for population extinction from the previous generation
    if len(genotypes) == 0:
        # If population died out, fill remaining generations in fitness_history with empty arrays
        for _ in range(g, generation):
            fitness_history.append(np.array([]))
        break # Stop simulation

    # `population_fitness` from the *previous* step (initial or end of last generation)
    # now serves as the `lambdas` for this generation's reproduction.
    lambdas = population_fitness # These are the effective reproductive rates

    # If all lambdas are zero, no offspring will be produced, leading to extinction
    if np.all(lambdas == 0):
        # Set genotypes to empty to trigger extinction handling in next check
        genotypes = np.empty((0, loci))
        current_pop_size = 0
        population_fitness = np.array([]) # Population went extinct
        fitness_history.append(population_fitness.copy())
        # Fill remaining generations as empty
        for _ in range(g + 1, generation):
            fitness_history.append(np.array([]))
        break

    offspring_counts = np.random.poisson(lambdas)

    # --- Generate the next generation (offspring) ---
    new_gen_genotypes_list = []
    offspring_mutation_details = []
    offspring_counter = 0

    current_parent_population_size = len(genotypes) # Size of the current parent population

    for parent_idx in range(current_parent_population_size):
        parent_genotype = genotypes[parent_idx]
        num_offspring = offspring_counts[parent_idx]

        for _ in range(num_offspring):
            mutated_individual, mutated_loci_info = mutate(parent_genotype, mutation_rate)
            new_gen_genotypes_list.append(mutated_individual)

            if mutated_loci_info:
                offspring_mutation_details.append({
                    'offspring_idx': offspring_counter,
                    'parent_idx': parent_idx,
                    'mutations': mutated_loci_info
                })
            offspring_counter += 1

    # Update genotypes to be the new generation
    if new_gen_genotypes_list:
        genotypes = np.array(new_gen_genotypes_list)
    else:
        genotypes = np.empty((0, loci)) # Handle case with no offspring

    current_pop_size = len(genotypes) # Update current_pop_size for next fitness calculation

    # Recalculate 'population_fitness' for the *newly generated population*
    # This updated 'population_fitness' will be used as `lambdas` in the *next* iteration (g+1)
    if current_pop_size > 0:
        trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])
        population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection, current_pop_size, carrying_capacity, base_reproduction_rate)
    else:
        population_fitness = np.array([]) # Population went extinct

    fitness_history.append(population_fitness.copy()) # Store fitness of this new generation

    # (Original print statements removed for brevity and because they were commented out)
    #print(f"\n{'='*30} GENERATION {g+1} {'='*30}")
    #if current_pop_size > 0:
        #print(f"\n--- GENERATION {g+1}: New Generation Genotypes with Mutations ---")
        #for i, individual_genotype in enumerate(genotypes):
            ##print(f"Offspring {i}: Genotype = {individual_genotype}")
            #pass
        #print(f"\nGENERATION {g+1}: Recalculated Trait values for new generation ({current_pop_size} individuals):")
        #print(trait_values)
        #print(f"\nGENERATION {g+1}: Recalculated Fitness for new generation ({current_pop_size} individuals):")
        #print(population_fitness)
    #else:
        #print("No offspring were generated in this generation.")
        #print("Cannot recalculate trait values or fitness as no offspring were generated.")

In [100]:
#Plotting the distribution of fitness values for the final generation

final_generation_fitness = fitness_history[-1]

if len(final_generation_fitness) > 0:
    plt.figure(figsize=(8, 6))
    ns.histplot(data=final_generation_fitness, bins=10, kde=True, color="lightblue")
    # Add rugplot to show individual fitness scores
    ns.rugplot(a=final_generation_fitness, color="darkblue", height=0.05)
    plt.title(f"Fitness Distribution at Generation {generation} with Individual Scores")
    plt.xlabel("Frequency / Individual Scores")
    plt.ylabel("Fitness Value")
    plt.show()
else:
    print(f"Population went extinct before or at Generation {generation}. No fitness data to plot.")

Population went extinct before or at Generation 100. No fitness data to plot.
